# Camada Silver
Nessa etapa trataremos as tabelas da etapa bronze, definindo tipos, lidando com nulos, eliminando dados invalidos e demais transformações necessárias

## VRA
Vamos inspecionar a tabela `mvp_engenharia_dados.bronze.bronze_vra` para determinar quais tratamentos serão necessàrios

In [0]:
%sql
DESCRIBE mvp_engenharia_dados.bronze.bronze_vra

### Duplicidade de dados

In [0]:
from pyspark.sql.functions import col

vra = spark.table("mvp_engenharia_dados.bronze.bronze_vra")

total_rows = vra.count()
distinct_rows = vra.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Total de linhas: {total_rows:,d}")
print(f"Linhas distintas: {distinct_rows:,d}")
print(f"Duplicatas: {duplicates:,d} ({duplicates / total_rows * 100:.2f}%)")

if duplicates > 0:
    print("\nTop 10 linhas duplicadas:")
    display(
        vra.groupBy(vra.columns)
          .count()
          .filter(col("count") > 1)
          .orderBy(col("count").desc())
          .limit(10)
    )

In [0]:
vra = vra.drop_duplicates()

### Valores Nulos ou em Branco

In [0]:
from pyspark.sql.functions import col, when, count, trim

total = vra.count()

null_row = vra.select([count(when(col(c).isNull(), c)).alias(c) for c in vra.columns]).collect()[0]
blank_row = vra.select([count(when((col(c).isNotNull()) & (trim(col(c)) == ""), c)).alias(c) for c in vra.columns]).collect()[0]

results = [(c, int(null_row[c]), int(blank_row[c])) for c in vra.columns]
display(spark.createDataFrame(results, ["column", "null_count", "blank_count"]))

> Nao ha valores em branco (''), somente nulos, o que simplifica o tratamento.

#### Tipo Linha nulos

In [0]:
%sql
select * from mvp_engenharia_dados.bronze.bronze_vra where `Código Tipo Linha` is null

> Voos com Código Tipo Linha em branco estao todos **CANCELADOS**.  
> Como de qualquer forma esses voos nao serao considerados, podemos ignora-los 

In [0]:
vra = vra.dropna(subset="Código Tipo Linha")

#### Partida Prevista nulos

In [0]:
display(vra.filter(col("Partida Prevista").isNull()).limit(25))

> Os voos sem Partida Prevista (Ou chegada Prevista), não devem ser contabilizados para o fim de pontualidade.  
> Eles distorceriam o índice, melhorando o resultado artificialmente

In [0]:
vra = vra.dropna(subset=["Partida Prevista", "Chegada Prevista"], how='any')

#### Partida Real e Chegada Real
Vamos verificar se as partidas realizadas obrigatoriamente tem chegadas realizadas (e vice versa)

In [0]:
%sql
SELECT
	*

FROM
	mvp_engenharia_dados.bronze.bronze_vra

WHERE
	(
		`Partida Real` IS NULL
		AND `Chegada Real` IS NOT NULL
	)
	OR (
		`Partida Real` IS NOT NULL
		AND `Chegada Real` IS NULL
	)

> As partidas e chegadas realizadas sempre possuem par, ou seja, nunca temos somente a partida ou a chegada registrada somente, sempre temos ambas.
> Podemos filtrar tranquilamente as viagens sem partidas real registrada

In [0]:
vra = vra.dropna(subset=["Partida Real", "Chegada Real"], how='any')

### Justificativa
O campo Justificativa deixou de ser obrigatorio a partir de 2020, por isso está totalmente nulo na base.  
Vamos eliminar essa coluna

In [0]:
vra = vra.drop("Justificativa")

### Codeshare
Não é um campo relevante para nossa analise, então tambem pode ser eliminado

In [0]:
vra = vra.drop("Codeshare")

### Nulos pos tratamento

In [0]:
total_anterior = spark.table("mvp_engenharia_dados.bronze.bronze_vra").count()
total = vra.count()

null_row = vra.select([count(when(col(c).isNull(), c)).alias(c) for c in vra.columns]).collect()[0]
blank_row = vra.select([count(when((col(c).isNotNull()) & (trim(col(c)) == ""), c)).alias(c) for c in vra.columns]).collect()[0]

results = [(c, int(null_row[c]), int(blank_row[c])) for c in vra.columns]
display(spark.createDataFrame(results, ["column", "null_count", "blank_count"]))

print(f"Total de linhas Antes: {total_anterior}")
print(f"Total de linhas Atual: {total}")
print(f"Percentual Perdido: {(total_anterior - total) / total_anterior * 100:.2f}%")


> Sem mais nenhum valor nulo nas colunas.
> Foram eliminados um total de 7% de linhas

## Filtros de regras

Como só vamos considerar voos nacionais e não cancelados nas nossas métricas, podemos eliminar os demais voos da tabela.

### Voos Cancelados

In [0]:
vra.filter(col("Situação Voo") == "CANCELADO").count()

> Os filtros de nulo anteriores acabaram já eliminando os Cancelados

In [0]:
display(vra.groupBy("Situação Voo").count())

> Ao meu ver não faz sentido manter essa coluna se a tabela só irá possuir voos realizados

In [0]:
vra = vra.drop("Situação Voo")

### Tipos de Linhas
Como só queremos as linhas domésticas de passageiros, usaremos o tipo da linha para determinar.

| Código | Descrição |
| --- | --- |
| **N** | **Doméstica Mista: transporte de passageiros ou mistos, com todos os aeródromos em território brasileiro** |
| C | Doméstica Cargueira: transporte exclusivo de carga e/ou mala postal, com todos os aeródromos em território brasileiro |
| I | Internacional Mista: transporte de passageiros ou mistos, com ao menos um aeródromo em território estrangeiro |
| G | Internacional Cargueira: transporte exclusivo de carga e/ou rede postal, com ao menos um aeródromo em território estrangeiro |
| X | Não informado |

O que interessa para a nossa análise são as linhas do tipo **N**

In [0]:
display(vra.groupBy(col("Código Tipo Linha")).count())

In [0]:
vra =  vra.filter(col("Código Tipo Linha") == 'N')


E agora o campo se torna obsoleto

In [0]:
vra = vra.drop("Código Tipo Linha")

In [0]:
display(vra.limit(10))

## Código DI
O Código DI (Dígito Identificador) informa o tipo de etapa de voo.

| Código DI | Descrição |
| --- | --- |
| 0 | Etapa Regular |
| 2 | Etapa Extra |
| 3 | Etapa de Retorno |
| 4 | Inclusão de Etapa |
| 6 | Etapa Não Remunerada Sem Transporte de Objetos |
| 7 | Etapa de Voo de Fretamento |
| 9 | Etapa de Voo Charter |
| D | Etapa de Voo Duplicada |
| E | Etapa Não Remunerada Com Transporte de Objetos |

In [0]:
display(vra.groupBy("Código DI").count().sort(col("count").desc()))

In [0]:
from pyspark.sql.functions import when, lit

vra = vra.withColumn(
    "descricao_etapa",
    when(col("Código DI") == "0", lit("Etapa Regular"))
    .when(col("Código DI") == "2", lit("Etapa Extra"))
    .when(col("Código DI") == "3", lit("Etapa de Retorno"))
    .when(col("Código DI") == "4", lit("Inclusão de Etapa"))
    .when(col("Código DI") == "6", lit("Etapa Não Remunerada Sem Transporte de Objetos"))
    .when(col("Código DI") == "7", lit("Etapa de Voo de Fretamento"))
    .when(col("Código DI") == "9", lit("Etapa de Voo Charter"))
    .when(col("Código DI") == "D", lit("Etapa de Voo Duplicada"))
    .when(col("Código DI") == "E", lit("Etapa Não Remunerada Com Transporte de Objetos"))
    .otherwise(lit("Não identificado"))
)

display(vra.groupBy("Código DI", "descricao_etapa").count().orderBy(col("count").desc()))

Vamos manter apenas a descricao_etapa, e não mais o código

In [0]:
vra = vra.drop("Código DI")

## Campos que virão de dimensões
Vamos eliminar da tabela vra os campos que serão provenientes de tabelas de dimensão, como informaçao de empresa e aeroporto

In [0]:
vra = vra.drop("Empresa Aérea", "Descrição Aeroporto Origem", "Descrição Aeroporto Destino")

In [0]:
display(vra.limit(10))

## Normalizar os nomes das colunas da tabela
Utilizaremos a convencao snake_case

In [0]:
import re
import unicodedata

def to_snake_case(name):
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-zA-Z0-9]', '_', name).lower()
    name = re.sub(r'_+', '_', name).strip('_')
    return name

for old_name in vra.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        vra = vra.withColumnRenamed(old_name, new_name)

display(vra.limit(10))

## Tipos de dados
Vamos transformar para os tipos corretos. (datas, numeros...etc)

### Tipos numéricos
Só sobrou o numero_de_asentos como numérico, numero_voo por exemplo contem caracteres alfabéticos

In [0]:
vra = vra.withColumn("numero_de_assentos", col("numero_de_assentos").cast("int"))
display(vra.select("numero_de_assentos").describe())

> vale investigar esses voos com 0 assentos

In [0]:
display(vra.filter(col("numero_de_assentos") == 0).groupBy("descricao_etapa").count())

> Um misto de tipos de voos.
> Irei manter com a seguinte premissa: Esses voos embora não estejam com passageiros, podem afetar etapas subsequentes.

### Tipos de data hora e data

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, try_to_date, coalesce, col

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dtypes = dict(vra.dtypes)
if dtypes.get("partida_prevista") == "string":
    vra = vra.withColumn("partida_prevista", to_timestamp("partida_prevista", "dd/MM/yyyy HH:mm"))
if dtypes.get("partida_real") == "string":
    vra = vra.withColumn("partida_real", to_timestamp("partida_real", "dd/MM/yyyy HH:mm"))
if dtypes.get("chegada_prevista") == "string":
    vra = vra.withColumn("chegada_prevista", to_timestamp("chegada_prevista", "dd/MM/yyyy HH:mm"))
if dtypes.get("chegada_real") == "string":
    vra = vra.withColumn("chegada_real", to_timestamp("chegada_real", "dd/MM/yyyy HH:mm"))
if dtypes.get("referencia") == "string":
    vra = vra.withColumn("referencia", coalesce(try_to_date(col("referencia"), "dd/MM/yyyy HH:mm:ss"), try_to_date(col("referencia"), "yyyy-MM-dd")))

display(vra.limit(10))

## Novas colunas
Vamos adicionar novas colunas que podem ser uteis a analise, como data de partida/chegada e tempo de atraso

### Datas de partida e chegada

In [0]:
from pyspark.sql.functions import to_date

vra = (vra
    .withColumn("data_partida", to_date("partida_prevista"))
    .withColumn("data_chegada", to_date("chegada_prevista"))
)

display(vra.limit(10))

### Tempo de atraso (partida e chegada)

In [0]:
from pyspark.sql.functions import unix_timestamp, round

vra = (vra
    .withColumn("atraso_partida_min", round((unix_timestamp("partida_real") - unix_timestamp("partida_prevista")) / 60))
    .withColumn("atraso_chegada_min", round((unix_timestamp("chegada_real") - unix_timestamp("chegada_prevista")) / 60))
)

display(vra.limit(10))

## Categorias
Uma verificacao na situacao_partida e situacao_chegada, para confirmar que estao padronizadas

In [0]:
display(vra.groupBy("situacao_partida").count())
display(vra.groupBy("situacao_chegada").count())


> As categorias já seguem uma nomenclatura padronizada

## Persistir os dados da vra no schema silver

In [0]:
vra.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("mvp_engenharia_dados.silver.silver_vra")

print(f"Tabela silver_vra criada com {vra.count():,d} linhas e {len(vra.columns)} colunas")

In [0]:
%sql
select * from mvp_engenharia_dados.silver.silver_vra limit 100